# Fixture context — home vs away (`total_points`)
_Read-only: how much home advantage is worth in points and through which channel, by position. SGW rows only — `was_home` is ambiguous for double gameweeks._

**Sections:** (a) total_points home vs away · (b) blank & return rate home vs away

---

## Setup — and the SGW restriction
> Whole season, `minutes > 0`, **SGW rows only**; split by venue using `was_home`.

`was_home` is ambiguous for a double gameweek — a DGW player can play one fixture home and one away under a single flag and a doubled total — so DGW rows are excluded to avoid mislabelling venue; their treatment lives in `fixture_doubling.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.distribution import (
    compute_distribution_stats,
    compare_cohorts,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that was a predictive-evaluation
# choice in the older EDA-1 record, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. The 60-minute
# performance boundary is NOT imposed here -- deferred to the exposure/ layer.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

# CRITICAL: restrict to SGW so `was_home` is unambiguous. DGW rows can be one
# home + one away under a single was_home flag and a doubled total -> excluded.
df = df[df["fixture_context"] == "SGW"].copy()
df = df[df["was_home"].notna()].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]
VENUES = ["home", "away"]
df["venue"] = np.where(df["was_home"], "home", "away")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print("Venue analysis RESTRICTED to SGW rows (was_home unambiguous); DGW rows excluded.")
print(f"Population: minutes > 0, SGW only, n = {len(df):,} player-gameweeks")
print("Venue counts:")
for v in VENUES:
    print(f"  {v:<5}: {int((df['venue'] == v).sum()):>6,}")

Study range: GW 1 - GW 38 (whole season, from mart data_cutoff_gw)
Venue analysis RESTRICTED to SGW rows (was_home unambiguous); DGW rows excluded.
Population: minutes > 0, SGW only, n = 11,190 player-gameweeks
Venue counts:
  home :  5,591
  away :  5,599


## (a) `total_points` distribution home vs away by position
> How much is home advantage worth in points, and does it differ by position?

The `total_points` distribution for home and away player-gameweeks within each position via `compare_cohorts` (SGW only). The home-minus-away gap is the expected venue gain, position by position — descriptive association, not causal; the split is roughly balanced, so team-quality confounding is milder than for FDR.

In [2]:
va_rows = []
for pos in POSITIONS:
    sub = df[df.position == pos]
    cohorts = {v: sub[sub["venue"] == v] for v in VENUES}
    stats = compare_cohorts(cohorts, value_col="total_points")
    for v in VENUES:
        s = stats.loc[v]
        va_rows.append({
            "position": pos,
            "venue": v,
            "n": int(s["count"]) if not np.isnan(s["count"]) else 0,
            "mean": s["mean"],
            "median": s["median"],
            "std": s["std"],
            "p90": s["p90"],
        })
venue_by_pos = pd.DataFrame(va_rows)
display(venue_by_pos.round(2))

# Home-minus-away mean gap per position -- the headline home-advantage number.
gap_rows = []
for pos in POSITIONS:
    p = venue_by_pos[venue_by_pos.position == pos].set_index("venue")
    gap_rows.append({
        "position": pos,
        "mean_home": p.loc["home", "mean"],
        "mean_away": p.loc["away", "mean"],
        "home_minus_away": round(p.loc["home", "mean"] - p.loc["away", "mean"], 2),
    })
venue_gap = pd.DataFrame(gap_rows)
display(venue_gap.round(2))

,position,venue,n,mean,median,std,p90
0,GK,home,372,3.520,2.000,2.760,7.000
1,GK,away,375,3.170,2.000,2.640,7.000
2,DEF,home,1896,3.260,2.000,3.140,8.000
3,DEF,away,1949,2.820,2.000,3.090,7.000
4,MID,home,2616,3.040,2.000,2.970,7.000
5,MID,away,2592,2.780,2.000,2.700,6.000
6,FWD,home,707,3.020,2.000,3.270,8.000
7,FWD,away,683,2.880,2.000,2.980,7.000


,position,mean_home,mean_away,home_minus_away
0,GK,3.520,3.170,0.350
1,DEF,3.260,2.820,0.440
2,MID,3.040,2.780,0.270
3,FWD,3.020,2.880,0.140


## (b) Blank rate and return rate home vs away by position
> Does the home edge show up as fewer blanks, more returns, or both — and does the channel differ by position?

Within each (position, venue) cell, the share that blank (`= 0`) and the share that return (`>= 4`), SGW only. This decomposes the home-minus-away gap from (a) into floor (fewer blanks, likely clean-sheet-driven for DEF / GK) vs ceiling (more returns) — descriptive association over the participation population.

In [3]:
venue_rates = (
    df.groupby(["position", "venue"])["total_points"]
    .apply(lambda y: pd.Series({
        "n": len(y),
        "blank_rate_%": round((y == 0).mean() * 100, 1) if len(y) else float("nan"),
        "return_4+_%": round((y >= 4).mean() * 100, 1) if len(y) else float("nan"),
    }))
    .reset_index()
)
display(venue_rates)


,position,venue,level_2,total_points
0,DEF,away,n,1949.000
1,DEF,away,blank_rate_%,9.800
2,DEF,away,return_4+_%,27.800
3,DEF,home,n,1896.000
4,DEF,home,blank_rate_%,6.500
5,DEF,home,return_4+_%,36.200
6,FWD,away,n,683.000
7,FWD,away,blank_rate_%,2.500
8,FWD,away,return_4+_%,25.200
9,FWD,home,n,707.000
